## เริ่ม Model

## ใช้ Dataset ใน Sklearn มีหลายแบบโหลดไม่เหมือนกัน

## กรณีงานจริง กำหนด Feature และ Target

In [ ]:
# กำหนด features และ target
# X = df[["column1", "column2", "column3", "column4"]]  # features
# y = df["ผลลัพธ์"]    # target
# กำหนด target_names เอง
# target_names = ["cat", "dog", "rabbit"]

## เริ่ม dataset ตัวอย่าง

In [23]:
## Forest Covertypes dataset โหลดและแปลงกลับเป็น column ต้นทางที่ยังไม่ได้ one hot encoder
## Target classes (7 ชนิดของป่า)   581012 rows × 14 columns
## covertype คือ target
# Spruce/Fir ป่าสนสปรูซและเฟอร์ 
# Lodgepole Pine ป่าสน Lodgepole 
# Ponderosa Pine ป่าสน Ponderosa 
# Cottonwood/Willow ป่าต้น Cottonwood และ Willow 
# Aspen ป่า Aspen 
# Douglas-fir ป่าสน Douglas-f 
# Krummholz ป่าพุ่มไม้เตี้ยที่ขึ้นในพื้นที่สูงมาก 


In [11]:
import pandas as pd
forest = pd.read_csv(r'D:\Forest Covertypes.csv')
forest = forest.set_index('Index')

# forest.info()
forest.head(5)
# forest.index
# forest.columns
# forest.shape


,Elevation,Aspect,Slope,Horizontal_Distance_To_Hydrology,Vertical_Distance_To_Hydrology,Horizontal_Distance_To_Roadways,Hillshade_9am,Hillshade_Noon,Hillshade_3pm,Horizontal_Distance_To_Fire_Points,Wilderness_Area,Soil_Type,Cover_Type
Index,,,,,,,,,,,,,
0,2596,51,3,258,0,510,221,232,148,6279,Wilderness_Area_0,Soil_Type_28,Aspen
1,2590,56,2,212,-6,390,220,235,151,6225,Wilderness_Area_0,Soil_Type_28,Aspen
2,2804,139,9,268,65,3180,234,238,135,6121,Wilderness_Area_0,Soil_Type_11,Lodgepole Pine
3,2785,155,18,242,118,3090,238,238,122,6211,Wilderness_Area_0,Soil_Type_29,Lodgepole Pine
4,2595,45,2,153,-1,391,220,234,150,6172,Wilderness_Area_0,Soil_Type_28,Aspen


## สำรวจข้อมูลก่อนเข้า Model เพิ่มส่วน EDA อีกครั้ง

In [ ]:
# จัดการค่า Missing
# จัดการค่า Outlier
# จัดการค่า Dupplicate

forest.info()
# forest.isnull().sum()


## แยก Train กับ Test ที่ขั้นตอนนี้เลย

In [ ]:
from sklearn.model_selection import train_test_split

# แยก features และ target
X = forest.drop(columns=["Cover_Type"])   # ลบ target ออก
y = forest["Cover_Type"]                  # target

# แยก Train/Test 70:30  stratify=y รักษาสัดส่วน class ให้ข้อมูลแต่ละ Class เท่าต้นฉบับ
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42,stratify=y)

X_train
# y_train


## ต่อด้วย One hot Encoding หรือ Label Encoding

In [ ]:
## จัดการ column กลุ่ม Category และ Text ด้วย One hot Encoding
## ทำทั้ง Train set และ Test set
## ใช้ One hot Encoding เป็นมาตรฐาน ถึงจะเป็น label ที่มีลำดับหรือไม่มีลำดับก็ใช้ได้ดี
## ใช้ Label Encoding กับ Label ที่มีลำดับเท่านั้น

import pandas as pd
from sklearn.preprocessing import OneHotEncoder

# สมมติว่ามี categorical columns
cat_cols = ["Wilderness_Area", "Soil_Type"]

# ใช้ OneHotEncoder
# sparse_output=False เพื่อให้เป็น array ก่อนที่จะไปเข้า Dataframe เป็นหลาย column
# handle_unknown='ignore' ใส่กรณี test set มี category ใหม่ ที่ตอน train ไม่มี
# encoder = OneHotEncoder(handle_unknown='ignore',sparse_output=False)

encoder = OneHotEncoder(sparse_output=False)


# บน train .fit_transform
X_train_cat = encoder.fit_transform(X_train[cat_cols])

# แปลงกลับเป็น DataFrame พร้อม column names
# .get_feature_names_out() เพื่อแยก column ออกมา
#  index=X_train.index จับจาก index

X_train_cat_df = pd.DataFrame(X_train_cat,columns=encoder.get_feature_names_out(cat_cols),index=X_train.index)
X_train_final = pd.concat([X_train.drop(columns=cat_cols), X_train_cat_df], axis=1)


# บน test .transform
X_test_cat = encoder.transform(X_test[cat_cols])

# แปลงกลับเป็น DataFrame พร้อม column names
X_test_cat_df = pd.DataFrame(X_test_cat,columns=encoder.get_feature_names_out(cat_cols),index=X_test.index)
X_test_final = pd.concat([X_test.drop(columns=cat_cols), X_test_cat_df], axis=1)

X_train_final

# Feature Scaling

In [ ]:
# จัดการกลุ่ม Column ตัวเลข ที่อาจมีฐานข้อมูล หน่วย ไม่เท่ากัน
# ****เราทำ Feature Scaling เพื่อปรับหน่วยตัวเลขแต่ละ Column ให้ใกล้เคียงกัน****
#    ****กรณีหน่วยมันเป็นตัวเดียวกันอยู่แล้วไม่ต้องทำตีความยาก****

# standard normailize  เหมาะกับ Logistic Regression(Linear Classification) , SVM (Distance-based ใช้ระยะทาง)
# Min-Max normailize   เหมาะกับ Neural Network, KNN (Distance-based ใช้ระยะทาง Euclidean)
# RobustScaler normailize  เหมาะกับ Dataset ที่มี outlier เยอะๆ
# Treebase Model ไม่ต้อง feature Scaling Decision Tree, Random Forest, Gradient Boosting, XGBoost, LightGBM, CatBoost
## *****Tree-based models (Random Forest, XGBoost) **** → ไม่จำเป็นต้องทำ scaling เพราะไม่อ่อนไหวต่อสเกล

from sklearn.preprocessing import StandardScaler,MinMaxScaler,RobustScaler

# 10 column ที่เป็นตัวเลข Float มาปรับ Scale กัน
numeric_cols = ["Elevation", "Aspect", "Slope",
                "Horizontal_Distance_To_Hydrology",
                "Vertical_Distance_To_Hydrology",
                "Horizontal_Distance_To_Roadways",
                "Hillshade_9am", "Hillshade_Noon", "Hillshade_3pm",
                "Horizontal_Distance_To_Fire_Points"]

X_numtrain = X_train_final[numeric_cols]
X_catgorytrain = X_train_final.drop(columns = numeric_cols)

X_numtest = X_test_final[numeric_cols]
X_catgorytest = X_test_final.drop(columns = numeric_cols)


# X_catgory.columns
# X_num.columns
## แบบที่ 1 # Standardize numeric features
# scaler = StandardScaler()
# X_num_scaled = scaler.fit_transform(X_num)

## แบบที่ 2 # MinMaxScaler numeric features
# scaler = MinMaxScaler()
# X_num_scaled = scaler.fit_transform(X_num)

## แบบที่ 3 # RobustScaler  numeric features
# scaler = RobustScaler()
# X_num_scaled = scaler.fit_transform(X_num)

## ค่า x และ y
# X = pd.concat([pd.DataFrame(X_num_scaled, columns=numeric_cols), X_cat], axis=1)
# y = df["Cover_Type"]


## แยก 2 แบบก่อน RobustScaler ใช้กรณี outlier เยอะๆ

In [ ]:
## แบบที่ 1  Logistic Regression, SVM
## แบบที่ 1  บน Train set  .fit_transform

from sklearn.preprocessing import StandardScaler,MinMaxScaler,RobustScaler
scaler1 = StandardScaler()            # สร้างครั้งเดียวพอ
X_num_trainscaled1 = scaler1.fit_transform(X_numtrain)


# เอา X_num_scaled ที่แปลงแล้ว เข้า dataframe และรวมกับ X_catgory ที่ไม่รวม Cover_Type
#  index=X_num.index เป็นการบอกว่า Scaler ที่แปลงแล้วให้ใช้ index เดิม

X_num_trainscaled_df = pd.DataFrame(X_num_trainscaled1, columns=numeric_cols,index=X_numtrain.index)
xtrainscaler1 = pd.concat([X_num_trainscaled_df, X_catgorytrain], axis=1)

# target y
ytrainscaler1 = y_train

## ได้ที่จะใช้
# xtrainscaler1
# ytrainscaler1

In [ ]:
## แบบที่ 1  บน Test set  .transform
X_num_testscaled1 = scaler.transform(X_numtest)

X_num_testscaled_df = pd.DataFrame(X_num_testscaled1, columns=numeric_cols,index=X_numtest.index)
xtestscaler1 = pd.concat([X_num_testscaled_df, X_catgorytest], axis=1)

# target y
ytestscaler1 = y_test

## ได้ที่จะใช้
# xtestscaler1
# ytestscaler1


In [ ]:
## แบบที่ 2 Neural Network, KNN
## แบบที่ 2  บน Train set  .fit_transform

scaler2 = MinMaxScaler()
X_num_trainscaled = scaler2.fit_transform(X_numtrain)

# เอา X_num_scaled ที่แปลงแล้ว เข้า dataframe และรวมกับ X_catgory ที่ไม่รวม Cover_Type
#  index=X_num.index เป็นการบอกว่า Scaler ที่แปลงแล้วให้ใช้ index เดิม

X_num_trainscaled_df = pd.DataFrame(X_num_trainscaled, columns=numeric_cols,index=X_num.index)
xtrainscaler1 = pd.concat([X_num_trainscaled_df, X_catgorytrain], axis=1)

# target y
ytrainscaler1 = y_train

## บน Train set  .transform
scaler2 = MinMaxScaler()
X_num_testscaled = scaler2.transform(X_numtest)

X_num_testscaled_df = pd.DataFrame(X_num_testscaled, columns=numeric_cols,index=X_num.index)
xtestscaler1 = pd.concat([X_num_testscaled_df, X_catgorytrain], axis=1)

## Feature Engineering

In [ ]:
# สร้างฟีเจอร์ใหม่จากข้อมูลเดิม							
	วันเกิด → อายุ							
	ทำ binning เช่น อายุแบ่งเป็นช่วง (0–18, 19–35, 36–60, 60+) 							
	เป็น category กลุ่มอายุแต่ละ Gen							
	วันที่ซื้อสินค้า → วันในสัปดาห์, เดือน, ฤดูกาล			จันทร์ อังคาร พุธ พฤหัส ศุกร์ เสาร์ อาทิตย์				เดือน
	ที่อยู่ → รหัสไปรษณีย์, ภูมิภาค							
#  แปลงข้อมูลให้อยู่ในรูปที่โมเดลใช้ได้							
	ข้อความ → TF-IDF, Word Embedding			ใน NLP อันนี้ไม่ต้อง				
	หมวดหมู่ → One-hot encoding หรือ Label encoding							
#   รวมฟีเจอร์เพื่อสร้างข้อมูลเชิงลึก							
	รายได้ต่อเดือน ÷ จำนวนสมาชิกครอบครัว → รายได้ต่อหัว							
	ยอดขาย ÷ จำนวนวัน → ยอดขายเฉลี่ยต่อวัน							
#   ลด noise							
	ตัดค่า Outlier ที่ผิดปกติจริงๆออก มันเกิด noise							


## Feature Selection

In [ ]:
## ใช้วิธี Correlation

# สมมติว่า X_train เป็น DataFrame และ y_train เป็น Series ของ target
# รวม X_train และ y_train เข้าด้วยกันเพื่อคำนวณ correlation
df = xscaler1.copy()
df['target'] = yscaler1

df

# คำนวณ correlation matrix
# corr_matrix = df.corr()

# ดูความสัมพันธ์ของแต่ละ feature กับ target
# corr_with_target = corr_matrix['target'].drop('target').sort_values(key=abs, ascending=False)

# print("Correlation ของแต่ละ feature กับ target:")
# print(corr_with_target)


In [ ]:
## ใช้วิธี Embedding โยนเข้า Xgboost  เพื่อหา feature importance

# รอบแรก: train เพื่อดู feature importance
model = XGBClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

# ดู feature importance
importances = model.feature_importances_
feature_names = X_train.columns
important_features = [f for f, imp in zip(feature_names, importances) if imp > 0.01]

print("Selected features:", important_features)

# รอบสอง: train จริงด้วยเฉพาะ important features
X_train_sel = X_train[important_features]
X_test_sel = X_test[important_features]

model_final = XGBClassifier(n_estimators=500, random_state=42)
model_final.fit(X_train_sel, y_train)
print("Accuracy:", model_final.score(X_test_sel, y_test))